In [2]:
from __future__ import annotations

import os
import sys
from pathlib import Path

CHATBOT_ROOT = Path.cwd().resolve()
for _candidate in [CHATBOT_ROOT, *CHATBOT_ROOT.parents]:
    if (_candidate / "src" / "pipeline").exists():
        CHATBOT_ROOT = _candidate
        break
REPO_ROOT = CHATBOT_ROOT.parent
if str(CHATBOT_ROOT) not in sys.path:
    sys.path.insert(0, str(CHATBOT_ROOT))
print("CHATBOT_ROOT =", CHATBOT_ROOT)

from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env") 

from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextPrecisionWithoutReference

from run_ppr_hybrid_question import run_ppr_hybrid

CHATBOT_ROOT = /Users/senghok/Projects/gds_tender_test/chatbot


/opt/anaconda3/envs/gds_test/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
EVAL_MODEL = os.getenv("RAGAS_EVAL_MODEL", "gpt-4o-mini")

DEFAULT_QUESTION = (
    "How is the combined pipe rack – equipment module connected to the "
    "compressor shelter in the proposed construction sequence?"
)

In [4]:
def get_evaluator_llm():
    """Plain OpenAI (not this app's Azure/APIM setup), authenticated via
    OPENAI_API_KEY. Must be an async client -- see the markdown cell above."""
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("OPENAI_API_KEY not set (checked process env and .env at repo root)")
    async_client = AsyncOpenAI(api_key=api_key)
    return llm_factory(EVAL_MODEL, provider="openai", client=async_client)


evaluator_llm = get_evaluator_llm()
print(f"evaluator_llm ready (model={EVAL_MODEL!r})")

evaluator_llm ready (model='gpt-4o-mini')


## Toy example (Eiffel Tower)

The example from the ragas docs -- validates the setup works at all before trusting it against real pipeline output.

In [4]:
async def test_toy_example(evaluator_llm) -> None:
    print("=" * 100)
    print("Toy example (Eiffel Tower)")
    print("=" * 100)

    metric = ContextPrecisionWithoutReference(llm=evaluator_llm)
    result = await metric.ascore(
        user_input="Where is the Eiffel Tower located?",
        response="The Eiffel Tower is located in Paris.",
        retrieved_contexts=["The Eiffel Tower is located in Paris."],
    )
    print(f"ContextPrecisionWithoutReference: {result.value}")
    return result

In [5]:
toy_result = await test_toy_example(evaluator_llm)

Toy example (Eiffel Tower)
ContextPrecisionWithoutReference: 0.9999999999


## Real pipeline question

Runs one question through the PPR + Hybrid pipeline (`run_ppr_hybrid_question.py`) and scores its actual retrieved contexts + answer. Change `question` in the run cell and re-run to try others without rebuilding `evaluator_llm`.

In [6]:
async def test_real_question(evaluator_llm, question: str):
    print("=" * 100)
    print(f"Real pipeline question: {question}")
    print("=" * 100)

    result = run_ppr_hybrid(question)
    answer = result["answer"]
    contexts = [c.get("content", "") for c in result["reranked_chunks"] if c.get("content")]
    print(f"Retrieved contexts: {len(contexts)}")
    print(f"Answer: {answer.answer[:300]}")
    print()

    metric = ContextPrecisionWithoutReference(llm=evaluator_llm)
    score = await metric.ascore(
        user_input=question, response=answer.answer, retrieved_contexts=contexts,
    )
    print(f"ContextPrecisionWithoutReference: {score.value}")
    return {"answer": answer, "contexts": contexts, "score": score}

In [9]:
question = DEFAULT_QUESTION
real_result = await test_real_question(evaluator_llm, question)

Real pipeline question: How is the combined pipe rack – equipment module connected to the compressor shelter in the proposed construction sequence?


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 6626.77it/s]


Retrieved contexts: 20
Answer: The combined pipe rack – equipment module is connected to the compressor shelter in the proposed construction sequence through modular integration and hook-up spools. The North-South pipe rack is integrated into the AGRU module, which is part of the redesigned layout. This integration allows for dec

ContextPrecisionWithoutReference: 0.8901527599362564


## Batch eval: all questions in `data/eval_multihop_100_ppr.jsonl`

Runs every question through the PPR + Hybrid pipeline and scores each with `ContextPrecisionWithoutReference`, sequentially (`run_ppr_hybrid`'s cross-encoder reranker isn't thread-safe on this machine -- same constraint as every other pipeline eval script in `scripts/domain_eval/`). Each question runs the full pipeline + a ragas LLM call, so this will take a while for all 86 -- set `LIMIT` below to a small number first to sanity-check before the full run.

In [5]:
import json

QUESTIONS_PATH = REPO_ROOT / "data" / "eval_multihop_100_ppr.jsonl"
RESULTS_PATH = CHATBOT_ROOT / "domain_data" / "eval_results" / "ragas_context_precision_multihop_eval.json"


def load_eval_questions(limit=None):
    rows = []
    with QUESTIONS_PATH.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows[:limit] if limit else rows


LIMIT = None  # set e.g. 5 for a quick smoke test before the full 86-question run
eval_questions = load_eval_questions(limit=LIMIT)
print(f"{len(eval_questions)} questions loaded from {QUESTIONS_PATH}")

86 questions loaded from /Users/senghok/Projects/gds_tender_test/data/eval_multihop_100_ppr.jsonl


In [6]:
async def run_batch_ragas_eval(evaluator_llm, questions):
    """Sequential -- run_ppr_hybrid uses a cross-encoder reranker that isn't
    thread-safe on this machine, so questions are scored one at a time, not
    concurrently (same constraint as every other pipeline eval script in
    scripts/domain_eval/). A single question's failure is caught and logged,
    not fatal to the whole batch."""
    metric = ContextPrecisionWithoutReference(llm=evaluator_llm)
    results = []
    for i, item in enumerate(questions, 1):
        question = item["question"]
        print(f"[{i}/{len(questions)}] {question[:80]}")
        try:
            result = run_ppr_hybrid(question)
            answer = result["answer"]
            contexts = [c.get("content", "") for c in result["reranked_chunks"] if c.get("content")]
            score = await metric.ascore(
                user_input=question, response=answer.answer, retrieved_contexts=contexts,
            )
            results.append({
                "id": i,
                "question": question,
                "reference_answer": item.get("reference_answer"),
                "question_type": item.get("question_type"),
                "n_hops": item.get("n_hops"),
                "ok": True,
                "answer": answer.answer,
                "n_contexts": len(contexts),
                "context_precision_without_reference": score.value,
            })
            print(f"    context_precision_without_reference={score.value:.4f}")
        except Exception as exc:
            print(f"    FAILED: {type(exc).__name__}: {exc}")
            results.append({
                "id": i, "question": question, "ok": False,
                "error": f"{type(exc).__name__}: {exc}",
            })
    return results

In [8]:
batch_results = await run_batch_ragas_eval(evaluator_llm, eval_questions[27:])

[1/59] Why is it important to supply seamless pipes and fittings for the piping network


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 6200.90it/s]


    context_precision_without_reference=0.7098
[2/59] How is the Human Resources and Industrial Relations Strategy connected to the Co
    context_precision_without_reference=1.0000
[3/59] Which supplier of manual call points is also an approved vendor for water system
    context_precision_without_reference=0.6722
[4/59] Where can the amended terms for Section 18 – inclement Weather, which changed 'w
    context_precision_without_reference=0.8333
[5/59] If the Cyber Security Management Plan is achieved 12 weeks after the Contract Da
    context_precision_without_reference=1.0000
[6/59] How can changes to the Work, authorized through a Change Order, impact the cost 
    context_precision_without_reference=1.0000
[7/59] Which vendor supplies both Thermal Mass Flow Meters and Abrasive Service Sequenc
    context_precision_without_reference=0.5687
[8/59] If the Semi-Lean/Lean H-EX L-552-E-001-C is scheduled for completion 2 weeks aft
    context_precision_without_reference=0.9967
[9/59] W

In [ ]:
ok_results = [r for r in batch_results if r.get("ok")]
scores = [r["context_precision_without_reference"] for r in ok_results]
avg_score = sum(scores) / len(scores) if scores else None

summary = {
    "n_questions": len(batch_results),
    "n_ok": len(ok_results),
    "n_errors": len(batch_results) - len(ok_results),
    "avg_context_precision_without_reference": round(avg_score, 4) if avg_score is not None else None,
}

print("=" * 100)
print("BATCH RAGAS EVAL -- SUMMARY")
print("=" * 100)
for k, v in summary.items():
    print(f"{k:<45}{v}")

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.write_text(
    json.dumps({"summary": summary, "results": batch_results}, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(f"\nWrote full detail -> {RESULTS_PATH}")